In [0]:
# Import required PySpark functions
from pyspark.sql.functions import (
    current_timestamp, regexp_replace, trim, col, when, lit,
    row_number, monotonically_increasing_id, max as spark_max, 
    first, last, sum, to_date, to_timestamp
)
from pyspark.sql.window import Window
from pyspark.sql.types import BooleanType, DoubleType, IntegerType

In [0]:
# Create silver schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS 02_silver.clean")

In [0]:
def clean_bronze_to_silver(table_name):
    df = spark.table(f"01_bronze.raw.{table_name}")
    initial_count = df.count()
    df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])
    for column in df.columns:
        df = df.withColumn(column, trim(col(column)))
    for column in df.columns:
        df = df.withColumn(column, 
            when((col(column) == "") | (col(column) == " "), lit(None))
            .otherwise(col(column)))
    before_dedup = df.count()
    df = df.dropDuplicates()
    after_dedup = df.count()
    if before_dedup != after_dedup:
        print(f" Removed {before_dedup - after_dedup} duplicate rows")
    else:
        print("No duplicate rows found")
    id_column = f"{table_name}_id"
    if id_column in df.columns:
        null_count = df.filter(col(id_column).isNull()).count()
        if null_count > 0:
            df = df.withColumn("_original_order", monotonically_increasing_id())
            window_spec = Window.orderBy("_original_order").rowsBetween(Window.unboundedPreceding, 0)
            df = df.withColumn("_last_valid_id", 
                last(col(id_column), ignorenulls=True).over(window_spec))
            df = df.withColumn("_null_count",
                when(col(id_column).isNull(), 1).otherwise(0))
            df = df.withColumn("_cumsum_nulls",
                when(col(id_column).isNotNull(), 0)
                .otherwise(sum(col("_null_count")).over(window_spec)))
            df = df.withColumn(id_column,
                when(col(id_column).isNull(),
                    when(col("_last_valid_id").isNotNull(),
                         (col("_last_valid_id").cast("int") + col("_cumsum_nulls")).cast("string"))
                    .otherwise((col("_cumsum_nulls")).cast("string")))
                .otherwise(col(id_column)))
            df = df.drop("_original_order", "_last_valid_id", "_null_count", "_cumsum_nulls")
    for column in df.columns:
        if "is_active" in column.lower():
            print(f"  Converting {column} to boolean...")
            df = df.withColumn(column,
                when(col(column).isin("1", "Yes", "Y", "true", "True", "TRUE"), True)
                .when(col(column).isin("0", "No", "N", "false", "False", "FALSE"), False)
                .otherwise(None).cast(BooleanType()))
    if table_name == "opportunity":
        if "revenue_amount" in df.columns:
            df = df.withColumn("revenue_amount", 
                regexp_replace(col("revenue_amount"), "[^0-9.\\-]", ""))
            df = df.withColumn("revenue_amount", 
                when((col("revenue_amount").isNotNull()) & (col("revenue_amount") != ""),
                     col("revenue_amount").cast(DoubleType()))
                .otherwise(lit(0.0)))
    if table_name == "product":
        if "list_price" in df.columns:
            df = df.withColumn("list_price",
                when((col("list_price").isNotNull()) & (col("list_price") != ""),
                     col("list_price").cast(DoubleType()))
                .otherwise(lit(0.0)))
    df = df.withColumn("processed_time", current_timestamp())
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
      .saveAsTable(f"02_silver.clean.{table_name}")

In [0]:

df_emp = spark.table("02_silver.clean.employee")
total_rows = df_emp.count()
df_emp_all = df_emp.select("employee_id", "employee_name", "role", "region", "is_active_flag") \
    .orderBy(col("employee_id").cast("int"))
display(df_emp_all)


In [0]:
df_opp = spark.table("02_silver.clean.opportunity")
df_opp_sample = df_opp.select("opportunity_id", "customer_id", "product_id", "employee_id", "revenue_amount", "close_status", "contract_term", "start_date", "end_date") \
    .orderBy(col("opportunity_id").cast("int")) \
    .limit(150000)
display(df_opp_sample)

In [0]:

df_prod = spark.table("02_silver.clean.product")
print(f"\nTotal rows: {df_prod.count()}")
df_prod_all = df_prod.select("product_id", "product_name", "plan_name", "billing_cycle", "list_price", "is_active") \
    .orderBy("product_id")
display(df_prod_all)

In [0]:
df_cust = spark.table("02_silver.clean.customer")
total_rows = df_cust.count()
df_cust_all = df_cust.select("customer_id", "customer_name", "country", "industry_type", "account_created_date", "is_active") \
    .orderBy(col("customer_id").cast("int"))
display(df_cust_all)


In [0]:

df_fx = spark.table("02_silver.clean.fx_rate")
total_rows = df_fx.count()
df_fx_all = df_fx.select("*") \
    .orderBy("currency_code")
display(df_fx_all)


In [0]:

df_customer = spark.table("01_bronze.raw.customer")
for field in df_customer.schema.fields:
    if field.name in ["customer_id", "Account Created Date"]:
        print(f"  - {field.name}: {field.dataType}")
df_customer = df_customer.toDF(*[c.lower().replace(" ", "_") for c in df_customer.columns])
df_customer = df_customer \
    .withColumn("customer_id", col("customer_id").cast(IntegerType())) \
    .withColumn("account_created_date", to_date(col("account_created_date"), "dd-MM-yyyy"))
for field in df_customer.schema.fields:
    if field.name in ["customer_id", "account_created_date"]:
        print(f"  - {field.name}: {field.dataType}")
df_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("02_silver.clean.customer")

In [0]:
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import IntegerType, DateType
df_employee = spark.table("01_bronze.raw.employee")
for field in df_employee.schema.fields:
    if field.name in ["employee_id", "Hire Date", "Last Update"]:
        print(f"  - {field.name}: {field.dataType}")
df_employee = df_employee.toDF(*[c.lower().replace(" ", "_") for c in df_employee.columns])
before_filter = df_employee.count()
df_employee = df_employee.filter(col("employee_id").isNotNull())
after_filter = df_employee.count()
removed_rows = before_filter - after_filter
df_employee = df_employee \
    .withColumn("employee_id", col("employee_id").cast(IntegerType())) \
    .withColumn("hire_date", to_date(col("hire_date"), "dd-MM-yyyy")) \
    .withColumn("last_update", to_date(col("last_update"), "dd-MM-yyyy"))
df_employee = df_employee.withColumn(
    "is_active_flag",
    when(col("is_active_flag").isin("Y", "Yes", "1", "yes", "y", "true", "True", "TRUE"), True)
    .when(col("is_active_flag").isin("N", "No", "0", "no", "n", "false", "False", "FALSE"), False)
    .otherwise(None).cast(BooleanType())
)
for field in df_employee.schema.fields:
    if field.name in ["employee_id", "hire_date", "last_update", "is_active_flag"]:
        print(f"  - {field.name}: {field.dataType}")
df_employee.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("02_silver.clean.employee")

In [0]:

from pyspark.sql.functions import col, to_date
from pyspark.sql.types import FloatType, DateType
df_fx_rate = spark.table("01_bronze.raw.fx_rate")
for field in df_fx_rate.schema.fields:
    if field.name in ["FX Rate to GBP", "Effective Date"]:
        print(f"  - {field.name}: {field.dataType}")
df_fx_rate = df_fx_rate.toDF(*[c.lower().replace(" ", "_") for c in df_fx_rate.columns])
df_fx_rate = df_fx_rate \
    .withColumn("fx_rate_to_gbp", col("fx_rate_to_gbp").cast(FloatType())) \
    .withColumn("effective_date", to_date(col("effective_date"), "yyyy-MM-dd"))
for field in df_fx_rate.schema.fields:
    if field.name in ["fx_rate_to_gbp", "effective_date"]:
        print(f"  - {field.name}: {field.dataType}")
df_fx_rate.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("02_silver.clean.fx_rate")

In [0]:
from pyspark.sql.functions import col, to_date, to_timestamp, regexp_replace
from pyspark.sql.types import IntegerType, DateType, TimestampType, DoubleType
df_opportunity = spark.table("01_bronze.raw.opportunity")
for field in df_opportunity.schema.fields:
    if field.name in ["opportunity_id", "customer_id", "product_id", "employee_id", "Start Date", "End Date", "Created Timestamp", "Revenue Amount"]:
        print(f"  - {field.name}: {field.dataType}")
df_opportunity = df_opportunity.toDF(*[c.lower().replace(" ", "_") for c in df_opportunity.columns])
before_filter = df_opportunity.count()
df_opportunity = df_opportunity.filter(col("opportunity_id").isNotNull())
after_filter = df_opportunity.count()
removed_rows = before_filter - after_filter
df_opportunity = df_opportunity \
    .withColumn("opportunity_id", col("opportunity_id").cast(IntegerType())) \
    .withColumn("customer_id", col("customer_id").cast(IntegerType())) \
    .withColumn("product_id", col("product_id").cast(IntegerType())) \
    .withColumn("employee_id", col("employee_id").cast(IntegerType())) \
    .withColumn("start_date", to_date(col("start_date"), "dd-MM-yyyy")) \
    .withColumn("end_date", to_date(col("end_date"), "dd-MM-yyyy")) \
    .withColumn("created_timestamp", to_timestamp(col("created_timestamp"), "dd-MM-yyyy HH:mm"))
df_opportunity = df_opportunity \
    .withColumn("revenue_amount", regexp_replace(col("revenue_amount"), "[^0-9.\\-]", "")) \
    .withColumn("revenue_amount", col("revenue_amount").cast(DoubleType()))
for field in df_opportunity.schema.fields:
    if field.name in ["opportunity_id", "customer_id", "product_id", "employee_id", "start_date", "end_date", "created_timestamp", "revenue_amount"]:
        print(f"  - {field.name}: {field.dataType}")
df_opportunity.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("02_silver.clean.opportunity")

In [0]:
from pyspark.sql.functions import col, to_date, when
from pyspark.sql.types import IntegerType, DateType, BooleanType, DoubleType
df_product = spark.table("01_bronze.raw.product")
for field in df_product.schema.fields:
    if field.name in ["product_id", "Created Date", "Is Active", "List Price"]:
        print(f"  - {field.name}: {field.dataType}")

df_product = df_product.toDF(*[c.lower().replace(" ", "_") for c in df_product.columns])
before_filter = df_product.count()
df_product = df_product.filter(col("product_id").isNotNull())
after_filter = df_product.count()
removed_rows = before_filter - after_filter
df_product = df_product \
    .withColumn("product_id", col("product_id").cast(IntegerType())) \
    .withColumn("created_date", to_date(col("created_date"), "dd-MM-yyyy"))

df_product = df_product.withColumn(
    "is_active",
    when(col("is_active").isin("Y", "Yes", "1", "yes", "y", "true", "True", "TRUE"), True)
    .when(col("is_active").isin("N", "No", "0", "no", "n", "false", "False", "FALSE"), False)
    .otherwise(None).cast(BooleanType())
)
df_product = df_product.withColumn("list_price", col("list_price").cast(DoubleType()))
for field in df_product.schema.fields:
    if field.name in ["product_id", "created_date", "is_active", "list_price"]:
        print(f"  - {field.name}: {field.dataType}")
df_product.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("02_silver.clean.product")